# UC2 PhoBERT Vietnamese Sentiment — MLflow Pipeline

**Dataset:** UIT-VSFC (Vietnamese Students Feedback Corpus)
**Model:** vinai/phobert-base
**Platform:** MLflow trên GCP
**Training:** Google Colab GPU T4

---

In [1]:
!nvidia-smi

Sun May 10 19:28:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bước 2.3: Cài đặt thư viện

In [2]:
!pip install mlflow==2.19.0 transformers==4.44.0 datasets==2.14.0 accelerate scikit-learn torch peft==0.11.0

In [3]:
#!pip install mlflow==2.19.0 transformers==4.41.0 datasets accelerate scikit-learn torch

In [4]:
# Kiểm tra versions
import mlflow, transformers, torch
print(f"MLflow: {mlflow.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

MLflow: 2.19.0
Transformers: 4.44.0
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Bước 2.4: Kết nối MLflow Server trên GCP
📸 MINH CHỨNG: Chụp screenshot output "Kết nối thành công"

In [5]:
import mlflow

# External IP của GCP VM
MLFLOW_URI = "http://136.119.253.114:5000"

mlflow.set_tracking_uri(MLFLOW_URI)

# Test kết nối
experiments = mlflow.search_experiments()
print(f" Connect successfully! Current experiments: {len(experiments)}")
for exp in experiments:
    print(f"  - {exp.name}")

✅ Connect successfully! Current experiments: 2
  - UC1_MNIST_MLflow
  - Default


---
## PHẦN 3: LOAD dataset
### Bước 3.1: Load UIT-VSFC dataset

In [6]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Đường dẫn trực tiếp đến các file Parquet (đã kiểm tra tồn tại)
urls = {
    "train": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet"
}

print(" Đang nạp dữ liệu trực tiếp từ định dạng Parquet...")

try:
    # Đọc dữ liệu trực tiếp từ URL
    train_df = pd.read_parquet(urls["train"])
    val_df = pd.read_parquet(urls["validation"])
    test_df = pd.read_parquet(urls["test"])

    # Chuyển đổi thành DatasetDict để bạn làm tiếp bài học
    dataset = DatasetDict({
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(val_df),
        "test": Dataset.from_pandas(test_df)
    })

    print("\n THÀNH CÔNG RỰC RỠ!")
    print(dataset)

    # Kiểm tra phân bổ nhãn để khớp với yêu cầu trong ảnh của bạn
    print("\n Phân bổ nhãn (Train):")
    # Cột nhãn trong dataset này thường tên là 'sentiment'
    print(train_df['sentiment'].value_counts().sort_index())
    print("\nÝ nghĩa: 0=Negative, 1=Neutral, 2=Positive")

except Exception as e:
    print(f"\n Vẫn gặp lỗi: {e}")

📦 Đang nạp dữ liệu trực tiếp từ định dạng Parquet...

✅ THÀNH CÔNG RỰC RỠ!
DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 11426
    })
    validation: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 1583
    })
    test: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 3166
    })
})

📊 Phân bổ nhãn (Train):
sentiment
0    5325
1     458
2    5643
Name: count, dtype: int64

Ý nghĩa: 0=Negative, 1=Neutral, 2=Positive


### Bước 3.2: Dataset


In [7]:
# Xem 5 samples đầu tiên
for i in range(5):
    sample = dataset['train'][i]
    print(f"[{sample['sentiment']}] {sample['sentence'][:80]}...")

# Phân bố nhãn
print("\nPhân bố nhãn (Train):")
train_df = pd.DataFrame(dataset['train'])
print(train_df['sentiment'].value_counts())
print("\nÝ nghĩa: 0=Negative, 1=Neutral, 2=Positive")

[2] slide giáo trình đầy đủ ....
[2] nhiệt tình giảng dạy , gần gũi với sinh viên ....
[0] đi học đầy đủ full điểm chuyên cần ....
[0] chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy ....
[2] thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp ....

Phân bố nhãn (Train):
sentiment
2    5643
0    5325
1     458
Name: count, dtype: int64

Ý nghĩa: 0=Negative, 1=Neutral, 2=Positive


---
## PHẦN 4: TOKENIZATION VỚI PhoBERT
### Bước 4.1: Load PhoBERT tokenizer

In [8]:
from transformers import AutoTokenizer

# Load PhoBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# Test tokenizer
sample = "Giảng viên dạy rất nhiệt tình và dễ hiểu"
tokens = tokenizer(sample, padding="max_length", truncation=True, max_length=256)
print(f"Input text: {sample}")
print(f"Token IDs (first 20): {tokens['input_ids'][:20]}")
print(f"Attention mask (first 20): {tokens['attention_mask'][:20]}")
print(f"Total tokens: {sum(tokens['attention_mask'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Input text: Giảng viên dạy rất nhiệt tình và dễ hiểu
Token IDs (first 20): [0, 34199, 1430, 940, 59, 2515, 939, 6, 592, 563, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Attention mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Total tokens: 11


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Bước 4.2: Tokenize toàn bộ dataset

In [9]:
MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# Tokenize (mất ~1-2 phút)
print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, batch_size=1000)

# Rename cột sentiment → labels (HuggingFace Trainer yêu cầu)
tokenized_datasets = tokenized_datasets.rename_column("sentiment", "labels")

# Set format PyTorch
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization hoàn tất!")
print(f"  Train: {len(tokenized_datasets['train'])} samples")
print(f"  Columns: {tokenized_datasets['train'].column_names}")

Tokenizing dataset...


Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

✅ Tokenization hoàn tất!
  Train: 11426 samples
  Columns: ['sentence', 'labels', 'topic', 'input_ids', 'token_type_ids', 'attention_mask']


---
## PHẦN 5: FINE-TUNE PhoBERT + LOG MLflow
### Bước 5.1: Định nghĩa hàm training

In [10]:
import time
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

def train_one_run(lr, batch_size, epochs, run_name):
    """Fine-tune PhoBERT 1 lần, log kết quả lên MLflow server GCP."""

    mlflow.set_experiment("UC2_PhoBERT_MLflow")

    with mlflow.start_run(run_name=run_name):
        start_time = time.time()

        # --- Log params ---
        mlflow.log_param("model", "vinai/phobert-base")
        mlflow.log_param("dataset", "UIT-VSFC")
        mlflow.log_param("num_labels", 3)
        mlflow.log_param("max_length", MAX_LENGTH)
        mlflow.log_param("lr", lr)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("platform", "MLflow-GCP")
        mlflow.log_param("gpu", torch.cuda.get_device_name(0))

        # --- Load model ---
        model = AutoModelForSequenceClassification.from_pretrained(
            "vinai/phobert-base",
            num_labels=3,
        )

        # --- Training arguments ---
        training_args = TrainingArguments(
            output_dir="./results_uc2",
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            logging_steps=50,
            fp16=True,              # Mixed precision cho T4
            report_to="none",       # Tắt wandb
            save_total_limit=1,     # Giữ 1 checkpoint (tiết kiệm disk)
        )

        # --- Trainer ---
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_datasets["train"],
            eval_dataset=tokenized_datasets["validation"],
            compute_metrics=compute_metrics,
        )

        # --- Train ---
        print(f"  🚀 Training {run_name}...")
        trainer.train()

        # --- Evaluate trên test set ---
        test_results = trainer.evaluate(tokenized_datasets["test"])
        accuracy = test_results["eval_accuracy"]
        f1 = test_results["eval_f1"]
        pipeline_time = time.time() - start_time

        # --- Log metrics ---
        mlflow.log_metric("test_accuracy", accuracy)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("pipeline_time_seconds", pipeline_time)

        print(f"  ✅ {run_name}: Accuracy={accuracy:.4f} | F1={f1:.4f} | TC7={pipeline_time:.1f}s")
        return accuracy, f1, pipeline_time

### Bước 5.2: PHẦN 1 — Chạy lặp 3 lần (đo TC7)
⏱️ Thời gian ước tính: ~45-60 phút (3 runs × 15-20 phút/run trên T4)

📸 MINH CHỨNG: Chụp screenshot output kết quả TC7 (accuracy + thời gian 3 runs)

In [11]:
# ========================================
# PHẦN 1: Chạy 3 lần với config mặc định
# Mục đích: Đo thời gian trung bình (TC7)
# ========================================
print("=" * 60)
print(" PHẦN 1: Chạy lặp 3 lần — Đo TC7 (thời gian pipeline)")
print("=" * 60)

results_repeat = []
for i in range(1, 4):
    print(f"\n[Run {i}/3]")
    acc, f1, t = train_one_run(
        lr=2e-5,
        batch_size=16,
        epochs=3,
        run_name=f"PhoBERT_run{i}"
    )
    results_repeat.append({"run": i, "accuracy": acc, "f1": f1, "time": t})

# Tổng kết
print("\n" + "=" * 60)
print(" KẾT QUẢ TC7:")
avg_acc = np.mean([r['accuracy'] for r in results_repeat])
avg_f1 = np.mean([r['f1'] for r in results_repeat])
avg_time = np.mean([r['time'] for r in results_repeat])
print(f"  Accuracy TB: {avg_acc:.4f}")
print(f"  F1 TB: {avg_f1:.4f}")
print(f"  TC7 (thời gian TB): {avg_time:.1f}s")
print("=" * 60)

 PHẦN 1: Chạy lặp 3 lần — Đo TC7 (thời gian pipeline)

[Run 1/3]


2026/05/10 19:29:53 INFO mlflow.tracking.fluent: Experiment with name 'UC2_PhoBERT_MLflow' does not exist. Creating a new experiment.


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  🚀 Training PhoBERT_run1...


...

  ✅ PhoBERT_run2: Accuracy=0.9340 | F1=0.9311 | TC7=756.0s
🏃 View run PhoBERT_run2 at: http://136.119.253.114:5000/#/experiments/2/runs/6f49be2db80346a7a521ab1404c314f1
🧪 View experiment at: http://136.119.253.114:5000/#/experiments/2

[Run 3/3]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  🚀 Training PhoBERT_run3...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.225000,0.232458,0.938092,0.934523
2,0.182000,0.227394,0.945041,0.942992
3,0.145700,0.239743,0.947568,0.945277


  ✅ PhoBERT_run3: Accuracy=0.9340 | F1=0.9311 | TC7=786.8s
🏃 View run PhoBERT_run3 at: http://136.119.253.114:5000/#/experiments/2/runs/6fe2f494b4c045b180f3507b6c9a7868
🧪 View experiment at: http://136.119.253.114:5000/#/experiments/2

 KẾT QUẢ TC7:
  Accuracy TB: 0.9331
  F1 TB: 0.9300
  TC7 (thời gian TB): 762.1s


### Bước 5.3: PHẦN 2 — TC2 Config Sweep
⏱️ Thời gian ước tính: ~45-60 phút

⚠️ Nếu batch_size=32 bị OOM, giảm MAX_LENGTH từ 256 → 128

In [12]:
# ========================================
# PHẦN 2: TC2 — Kiểm tra tính linh hoạt cấu hình
# Chạy 3 bộ hyperparameter khác nhau
# ========================================
print("=" * 60)
print(" PHẦN 2: TC2 Config Sweep (3 configs)")
print("=" * 60)

tc2_configs = [
    {"lr": 1e-5, "batch_size": 16, "epochs": 3},
    {"lr": 2e-5, "batch_size": 16, "epochs": 3},
    {"lr": 3e-5, "batch_size": 32, "epochs": 3},
]

results_tc2 = []
for cfg in tc2_configs:
    run_name = f"TC2_lr{cfg['lr']}_batch{cfg['batch_size']}"
    print(f"\n[Config: {run_name}]")
    acc, f1, t = train_one_run(run_name=run_name, **cfg)
    results_tc2.append({"config": run_name, "accuracy": acc, "f1": f1, "time": t})

# Tổng kết TC2
print("\n" + "=" * 60)
print(" KẾT QUẢ TC2 SWEEP:")
for r in results_tc2:
    print(f"  {r['config']}: acc={r['accuracy']:.4f}, f1={r['f1']:.4f}, time={r['time']:.1f}s")
print("=" * 60)

 PHẦN 2: TC2 Config Sweep (3 configs)

[Config: TC2_lr1e-05_batch16]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  🚀 Training TC2_lr1e-05_batch16...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.234500,0.234324,0.934302,0.928352
2,0.206100,0.211039,0.943778,0.942058
3,0.166500,0.229319,0.942514,0.938920


  ✅ TC2_lr1e-05_batch16: Accuracy=0.9292 | F1=0.9270 | TC7=930.0s
🏃 View run TC2_lr1e-05_batch16 at: http://136.119.253.114:5000/#/experiments/2/runs/f3ce15000e6846ebb69d0c12170b8fa7
🧪 View experiment at: http://136.119.253.114:5000/#/experiments/2

[Config: TC2_lr2e-05_batch16]


...

  ✅ TC2_lr2e-05_batch16: Accuracy=0.9340 | F1=0.9311 | TC7=932.7s
🏃 View run TC2_lr2e-05_batch16 at: http://136.119.253.114:5000/#/experiments/2/runs/a7945265ba7949b49adf5ecafcfef447
🧪 View experiment at: http://136.119.253.114:5000/#/experiments/2

[Config: TC2_lr3e-05_batch32]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  🚀 Training TC2_lr3e-05_batch32...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.223000,0.204623,0.940619,0.936958
2,0.140300,0.213117,0.939987,0.941109
3,0.103700,0.229000,0.943778,0.940843


  ✅ TC2_lr3e-05_batch32: Accuracy=0.9340 | F1=0.9289 | TC7=827.3s
🏃 View run TC2_lr3e-05_batch32 at: http://136.119.253.114:5000/#/experiments/2/runs/6cba2860ddb84ff889a950a5f3ba1bbf
🧪 View experiment at: http://136.119.253.114:5000/#/experiments/2

 KẾT QUẢ TC2 SWEEP:
  TC2_lr1e-05_batch16: acc=0.9292, f1=0.9270, time=930.0s
  TC2_lr2e-05_batch16: acc=0.9340, f1=0.9311, time=932.7s
  TC2_lr3e-05_batch32: acc=0.9340, f1=0.9289, time=827.3s


---
## PHẦN 6: EXPORT KẾT QUẢ
### Bước 6.1: Export CSV kết quả

In [13]:
import pandas as pd

# Lấy tất cả runs từ experiment UC2
experiment = mlflow.get_experiment_by_name("UC2_PhoBERT_MLflow")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Chọn cột quan trọng
cols = [
    "run_id", "tags.mlflow.runName",
    "params.model", "params.lr", "params.batch_size", "params.epochs",
    "metrics.test_accuracy", "metrics.test_f1", "metrics.pipeline_time_seconds"
]
results = runs_df[[c for c in cols if c in runs_df.columns]]

# Hiển thị
print(results.to_string(index=False))

# Lưu file CSV
results.to_csv("mlflow_uc2_results.csv", index=False)
print("\n✅ Đã lưu: mlflow_uc2_results.csv")

# Download file về máy
from google.colab import files
files.download("mlflow_uc2_results.csv")

                          run_id tags.mlflow.runName       params.model params.lr params.batch_size params.epochs  metrics.test_accuracy  metrics.test_f1  metrics.pipeline_time_seconds
6cba2860ddb84ff889a950a5f3ba1bbf TC2_lr3e-05_batch32 vinai/phobert-base     3e-05                32             3               0.933986         0.928883                     827.330379
a7945265ba7949b49adf5ecafcfef447 TC2_lr2e-05_batch16 vinai/phobert-base     2e-05                16             3               0.933986         0.931070                     932.656414
f3ce15000e6846ebb69d0c12170b8fa7 TC2_lr1e-05_batch16 vinai/phobert-base     1e-05                16             3               0.929248         0.926963                     929.997750
6fe2f494b4c045b180f3507b6c9a7868        PhoBERT_run3 vinai/phobert-base     2e-05                16             3               0.933986         0.931070                     786.820012
6f49be2db80346a7a521ab1404c314f1        PhoBERT_run2 vinai/phobert-base    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## ✅ HOÀN THÀNH!

Kiểm tra MLflow UI: http://136.119.253.114:5000

### Minh chứng cần lưu:
- [ ] Screenshot nvidia-smi (GPU T4)
- [ ] Screenshot kết nối MLflow thành công
- [ ] Screenshot dataset phân bố 3 nhãn
- [ ] Screenshot kết quả TC7 (3 runs)
- [ ] Screenshot kết quả TC2 sweep
- [ ] Screenshot MLflow UI với 6 runs UC2
- [ ] File CSV kết quả (mlflow_uc2_results.csv)